# Federated GraphSAGE for Illicit Transaction Detection

## 1. Dataset Exploration

We use the Elliptic Bitcoin transaction dataset to investigate
graph-based illicit transaction detection.

The dataset contains transaction features, transaction
relationships, and known transaction labels.

In [1]:
import pandas as pd

In [7]:
classes_path = "data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"
features_path = "data/elliptic_bitcoin_dataset/elliptic_txs_features.csv"

classes = pd.read_csv(classes_path)

print("Shape:", classes.shape)
print("\nColumns:")
print(classes.columns.tolist())

print("\nFirst 5 rows:")
display(classes.head())

Shape: (203769, 2)

Columns:
['txId', 'class']

First 5 rows:


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


In [8]:
print(classes["class"].value_counts())

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [9]:
edges_path = "data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"

edges = pd.read_csv(edges_path)

print("Shape:", edges.shape)

print("\nColumns:")
print(edges.columns.tolist())

print("\nFirst 5 rows:")
display(edges.head())

Shape: (234355, 2)

Columns:
['txId1', 'txId2']

First 5 rows:


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [10]:
features_sample = pd.read_csv(
    features_path,
    header=None,
    nrows=5
)

print("Sample shape:", features_sample.shape)

print("\nFirst 10 columns:")
display(features_sample.iloc[:, :10])

print("\nLast 5 columns:")
display(features_sample.iloc[:, -5:])

Sample shape: (5, 167)

First 10 columns:


,0,1,2,3,4,5,6,7,8,9
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523



Last 5 columns:


,162,163,164,165,166
0,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,0.277775,0.326394,1.293750,0.178136,0.179117


In [11]:
feature_columns = (
    ["txId", "time_step"]
    + [f"feature_{i}" for i in range(1, 166)]
)

features_sample.columns = feature_columns

print("Number of columns:", len(features_sample.columns))
print("\nFirst 10 columns:")
print(features_sample.columns[:10].tolist())

print("\nLast 5 columns:")
print(features_sample.columns[-5:].tolist())

display(features_sample.head())

Number of columns: 167

First 10 columns:
['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8']

Last 5 columns:
['feature_161', 'feature_162', 'feature_163', 'feature_164', 'feature_165']


,txId,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [12]:
print("Classes shape:", classes.shape)

print("\nClass columns:")
print(classes.columns.tolist())

print("\nFirst 10 rows:")
display(classes.head(10))

print("\nClass distribution:")
print(classes["class"].value_counts())

Classes shape: (203769, 2)

Class columns:
['txId', 'class']

First 10 rows:


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown
5,230459870,unknown
6,230333930,unknown
7,230595899,unknown
8,232013274,unknown
9,232029206,2



Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


In [13]:
edges_path = "data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"

edges = pd.read_csv(edges_path)

print("Edges shape:", edges.shape)

print("\nColumn names:")
print(edges.columns.tolist())

print("\nFirst 10 edges:")
display(edges.head(10))

Edges shape: (234355, 2)

Column names:
['txId1', 'txId2']

First 10 edges:


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206
5,232344069,27553029
6,36411953,230405052
7,34194980,5529846
8,3881097,232457116
9,230409257,32877982


In [14]:
print("Number of edges:", len(edges))

print("Unique source transactions:", edges["txId1"].nunique())
print("Unique destination transactions:", edges["txId2"].nunique())

all_nodes = set(edges["txId1"]) | set(edges["txId2"])

print("Unique transactions appearing in edges:", len(all_nodes))

Number of edges: 234355
Unique source transactions: 166345
Unique destination transactions: 148447
Unique transactions appearing in edges: 203769


In [15]:
print("Minimum txId:", min(all_nodes))
print("Maximum txId:", max(all_nodes))

Minimum txId: 1076
Maximum txId: 403244581


In [16]:
# Count how many connections each transaction has

source_counts = edges["txId1"].value_counts()
destination_counts = edges["txId2"].value_counts()

print("Average outgoing connections:",
      source_counts.mean())

print("Average incoming connections:",
      destination_counts.mean())

print("\nMaximum outgoing connections:",
      source_counts.max())

print("Maximum incoming connections:",
      destination_counts.max())

Average outgoing connections: 1.4088490787219334
Average incoming connections: 1.578711594036929

Maximum outgoing connections: 472
Maximum incoming connections: 284


In [17]:
# Find some transactions with many connections

print("Transactions with the most outgoing connections:")
display(source_counts.head(10))

print("\nTransactions with the most incoming connections:")
display(destination_counts.head(10))

Transactions with the most outgoing connections:


txId1
2984918    472
89273      288
102570     122
3181       112
7952        99
1891081     95
143705      92
565334      90
488266      88
793584      82
Name: count, dtype: int64


Transactions with the most incoming connections:


txId2
43388675     284
68705820     247
30699343     241
96576418     239
225859042    212
279187194    211
234890810    199
196107869    188
43397277     182
68706499     178
Name: count, dtype: int64

In [18]:
features_sample = pd.read_csv(
    features_path,
    header=None,
    nrows=1000
)

features_sample.columns = feature_columns

print("Sample shape:", features_sample.shape)

print("\nData types:")
print(features_sample.dtypes.value_counts())

print("\nTime-step range in sample:")
print(
    features_sample["time_step"].min(),
    "to",
    features_sample["time_step"].max()
)

print("\nFeature statistics:")
display(
    features_sample[
        ["feature_1", "feature_2", "feature_3",
         "feature_4", "feature_5"]
    ].describe()
)

Sample shape: (1000, 167)

Data types:
float64    165
int64        2
Name: count, dtype: int64

Time-step range in sample:
1 to 1

Feature statistics:


,feature_1,feature_2,feature_3,feature_4,feature_5
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,-0.039740,-0.151794,-0.728515,0.125654,0.234641
std,0.889864,0.177285,0.782537,1.447287,8.231080
min,-0.172979,-0.210553,-1.756361,-0.121970,-0.063725
25%,-0.172017,-0.184668,-1.201369,-0.121970,-0.043875
50%,-0.167356,-0.184668,-1.201369,-0.121970,-0.043875
75%,-0.151507,-0.182456,-0.091383,-0.046932,-0.043875
max,13.222433,2.456149,1.573595,28.317246,260.090707


In [19]:
print("Feature txId type:", features_sample["txId"].dtype)
print("Class txId type:", classes["txId"].dtype)
print("Edge txId1 type:", edges["txId1"].dtype)
print("Edge txId2 type:", edges["txId2"].dtype)

Feature txId type: int64
Class txId type: int64
Edge txId1 type: int64
Edge txId2 type: int64


In [20]:
feature_ids = set(features_sample["txId"])

class_ids = set(classes["txId"])

print("Feature sample IDs:", len(feature_ids))
print("Class IDs:", len(class_ids))

print(
    "Feature sample IDs found in classes:",
    len(feature_ids & class_ids)
)

Feature sample IDs: 1000
Class IDs: 203769
Feature sample IDs found in classes: 1000


In [21]:
print("Loading full feature dataset...")

features = pd.read_csv(
    features_path,
    header=None
)

features.columns = feature_columns

print("Finished loading!")
print("Shape:", features.shape)

display(features.head())

Loading full feature dataset...
Finished loading!
Shape: (203769, 167)


,txId,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


In [22]:
print("Number of unique feature IDs:", features["txId"].nunique())

print(
    "All class IDs present in features:",
    set(classes["txId"]).issubset(set(features["txId"]))
)

print(
    "All edge transaction IDs present in features:",
    set(edges["txId1"]).issubset(set(features["txId"]))
    and
    set(edges["txId2"]).issubset(set(features["txId"]))
)

Number of unique feature IDs: 203769
All class IDs present in features: True
All edge transaction IDs present in features: True


In [23]:
# Keep only the actual numerical transaction features
feature_cols = [f"feature_{i}" for i in range(1, 166)]

X = features[feature_cols].copy()

print("Feature matrix shape:", X.shape)
print("Missing values:", X.isna().sum().sum())

Feature matrix shape: (203769, 165)
Missing values: 0


In [24]:
# Keep only transactions with known labels
labeled_classes = classes[classes["class"] != "unknown"].copy()

print("Labeled transactions:", len(labeled_classes))
print("\nLabel distribution:")
print(labeled_classes["class"].value_counts())

Labeled transactions: 46564

Label distribution:
class
2    42019
1     4545
Name: count, dtype: int64


In [25]:
# Combine transaction features with their known labels
labeled_data = features[
    features["txId"].isin(labeled_classes["txId"])
].merge(
    labeled_classes,
    on="txId",
    how="inner"
)

print("Labeled dataset shape:", labeled_data.shape)

print("\nColumns:")
print(labeled_data.columns.tolist()[:10], "...")

print("\nLabels:")
print(labeled_data["class"].value_counts())

Labeled dataset shape: (46564, 168)

Columns:
['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8'] ...

Labels:
class
2    42019
1     4545
Name: count, dtype: int64


In [49]:
# Create a copy of the labeled dataset
model_data = labeled_data.copy()

# Make sure the original class values are numeric
model_data["class"] = pd.to_numeric(
    model_data["class"],
    errors="coerce"
)

# Convert the original labels:
# class 2 = licit  -> 0
# class 1 = illicit -> 1
model_data["label"] = model_data["class"].map({
    2: 0,
    1: 1
})

print("Model dataset shape:", model_data.shape)

print("\nNew label distribution:")
print(model_data["label"].value_counts(dropna=False))

print("\nCheck for missing labels:")
print(model_data["label"].isna().sum())

Model dataset shape: (46564, 169)

New label distribution:
label
0    42019
1     4545
Name: count, dtype: int64

Check for missing labels:
0


In [50]:
X = model_data[feature_cols]
y = model_data["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX data type:")
print(X.dtypes.value_counts())

print("\ny data type:")
print(y.dtype)

X shape: (46564, 165)
y shape: (46564,)

X data type:
float64    165
Name: count, dtype: int64

y data type:
int64


In [51]:
print("Original class values:")
print(model_data["class"].value_counts(dropna=False))

print("\nMissing values in label:")
print(model_data["label"].isna().sum())

print("\nUnique label values:")
print(model_data["label"].unique())

Original class values:
class
2    42019
1     4545
Name: count, dtype: int64

Missing values in label:
0

Unique label values:
[0 1]


In [52]:
# Prepare final feature matrix and labels

X = model_data[feature_cols].copy()
y = model_data["label"].astype(int)

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts())

print("\nMissing feature values:", X.isna().sum().sum())
print("Missing labels:", y.isna().sum())

Feature matrix shape: (46564, 165)
Label vector shape: (46564,)

Label distribution:
label
0    42019
1     4545
Name: count, dtype: int64

Missing feature values: 0
Missing labels: 0


In [53]:
import numpy as np

# Create an index for every transaction
indices = np.arange(len(model_data))

# Separate transactions by class
licit_indices = indices[y.values == 0]
illicit_indices = indices[y.values == 1]

# Reproducible random generator
rng = np.random.default_rng(42)

# Shuffle each class
rng.shuffle(licit_indices)
rng.shuffle(illicit_indices)


# Split each class into 70% training,
# 15% validation, and 15% testing
def split_indices(class_indices):
    n = len(class_indices)

    train_end = int(0.70 * n)
    val_end = int(0.85 * n)

    train = class_indices[:train_end]
    validation = class_indices[train_end:val_end]
    test = class_indices[val_end:]

    return train, validation, test


# Split licit transactions
licit_train, licit_val, licit_test = split_indices(licit_indices)

# Split illicit transactions
illicit_train, illicit_val, illicit_test = split_indices(illicit_indices)


# Combine both classes
train_indices = np.concatenate([licit_train, illicit_train])
val_indices = np.concatenate([licit_val, illicit_val])
test_indices = np.concatenate([licit_test, illicit_test])


# Shuffle each final split
rng.shuffle(train_indices)
rng.shuffle(val_indices)
rng.shuffle(test_indices)


# Create final feature datasets
X_train = X.iloc[train_indices]
X_val = X.iloc[val_indices]
X_test = X.iloc[test_indices]

# Create final label datasets
y_train = y.iloc[train_indices]
y_val = y.iloc[val_indices]
y_test = y.iloc[test_indices]


# Display results
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training: (32594, 165) (32594,)
Validation: (6985, 165) (6985,)
Testing: (6985, 165) (6985,)

Training labels:
label
0    29413
1     3181
Name: count, dtype: int64

Validation labels:
label
0    6303
1     682
Name: count, dtype: int64

Testing labels:
label
0    6303
1     682
Name: count, dtype: int64


# 2. Transaction Graph Construction

The Elliptic dataset represents Bitcoin transactions as a graph.

Each transaction is represented as a node, while the transaction relationships
in the edgelist are represented as edges.

The graph contains all transactions, including transactions whose labels are
unknown. Known labels are used for supervised training and evaluation, while
unlabeled transactions can still provide graph neighborhood information.

## 2.1 Mapping Transaction IDs to Node Indices

The original transaction IDs are large integer values. Graph learning models
work more conveniently with consecutive node indices such as 0, 1, 2, ... .

We therefore create a mapping from each transaction ID to a consecutive
integer node index.

In [54]:
# Create a mapping from transaction ID to consecutive node index

all_tx_ids = features["txId"].to_numpy()

tx_id_to_index = {
    tx_id: index
    for index, tx_id in enumerate(all_tx_ids)
}

print("Number of graph nodes:", len(tx_id_to_index))
print("First 5 mappings:")

for tx_id, index in list(tx_id_to_index.items())[:5]:
    print(tx_id, "->", index)

Number of graph nodes: 203769
First 5 mappings:
230425980 -> 0
5530458 -> 1
232022460 -> 2
232438397 -> 3
230460314 -> 4


## 2.2 Converting Transaction Relationships into Graph Edges

The edgelist contains pairs of transaction IDs representing relationships
between transactions.

We convert these transaction IDs into the consecutive node indices created
above.

In [55]:
# Convert transaction IDs in the edgelist to graph node indices

edge_source = edges["txId1"].map(tx_id_to_index)
edge_target = edges["txId2"].map(tx_id_to_index)

print("Missing source mappings:", edge_source.isna().sum())
print("Missing target mappings:", edge_target.isna().sum())

Missing source mappings: 0
Missing target mappings: 0


In [56]:
# Convert edges to integer node indices

edge_source = edge_source.astype(np.int64)
edge_target = edge_target.astype(np.int64)

# Create an undirected edge list by adding both directions
edge_source_undirected = np.concatenate([edge_source, edge_target])
edge_target_undirected = np.concatenate([edge_target, edge_source])

print("Original directed edges:", len(edge_source))
print("Undirected edge connections:", len(edge_source_undirected))

Original directed edges: 234355
Undirected edge connections: 468710


## 2.3 Preparing Node Features

Each transaction contains 165 numerical features.

These features will be used by GraphSAGE as the initial representation of
each transaction node.

In [57]:
# Prepare the node feature matrix

node_features = features[feature_cols].to_numpy(dtype=np.float32)

print("Node feature shape:", node_features.shape)
print("Number of nodes:", node_features.shape[0])
print("Number of features per node:", node_features.shape[1])

Node feature shape: (203769, 165)
Number of nodes: 203769
Number of features per node: 165


## 2.4 Preparing Node Labels

Only some transactions have known labels.

We therefore create a label array for the complete graph and use `-1` for
transactions whose labels are unknown.

The known labels are:

- 0 = Licit
- 1 = Illicit
- -1 = Unknown

In [58]:
# Create labels for every transaction in the graph
# -1 represents an unknown label

node_labels = np.full(len(features), -1, dtype=np.int64)

# Create a mapping from transaction ID to binary label
label_mapping = dict(
    zip(model_data["txId"], model_data["label"])
)

# Assign known labels to the corresponding graph nodes
for tx_id, label in label_mapping.items():
    node_index = tx_id_to_index[tx_id]
    node_labels[node_index] = label

print("Licit nodes:", np.sum(node_labels == 0))
print("Illicit nodes:", np.sum(node_labels == 1))
print("Unknown nodes:", np.sum(node_labels == -1))

Licit nodes: 42019
Illicit nodes: 4545
Unknown nodes: 157205


## 2.5 Graph Data Verification

Before training the GraphSAGE model, we verify that the graph contains the
expected number of nodes, edges, features, and labels.

In [61]:
# Final graph verification

print("GRAPH VERIFICATION")
print("Number of nodes:", len(features))
print("Number of edges:", len(edge_source_undirected))
print("Node feature matrix:", node_features.shape)

print("\nLABELS")
print("Licit:", np.sum(node_labels == 0))
print("Illicit:", np.sum(node_labels == 1))
print("Unknown:", np.sum(node_labels == -1))

print("\nEDGE RANGE")
print("Smallest node index:", edge_source_undirected.min())
print("Largest node index:", edge_source_undirected.max())

GRAPH VERIFICATION
Number of nodes: 203769
Number of edges: 468710
Node feature matrix: (203769, 165)

LABELS
Licit: 42019
Illicit: 4545
Unknown: 157205

EDGE RANGE
Smallest node index: 0
Largest node index: 203768
